\*Please Run in colab

# Setup

### Environment Setup

In [1]:
!pip install instructor

In [2]:
!cd /content
!rm -rf macro_financial_forecasting

In [3]:
!git clone https://github.com/chuanbinp/macro_financial_forecasting.git

Cloning into 'macro_financial_forecasting'...
remote: Enumerating objects: 2024, done.
remote: Counting objects: 100% (734/734), done.
remote: Compressing objects: 100% (272/272), done.
remote: Total 2024 (delta 558), reused 486 (delta 462), pack-reused 1290 (from 2)
Receiving objects: 100% (2024/2024), 39.77 MiB | 16.47 MiB/s, done.
Resolving deltas: 100% (1300/1300), done.


In [4]:
%cd macro_financial_forecasting/applications/macro_financial_forecasting/src

/content/macro_financial_forecasting/applications/macro_financial_forecasting/src


### Code Setup

In [6]:
from config import Config
from train_data_loader import TrainDataLoader
from data_model.bloomberg_news_entry import BloombergNewsEntry
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
config = Config("../config.env")
print(f"Config: {config}")

Config: Config(
  gemini_api_key: !secret!
  openai_api_key: !secret!
  llm_model: openai/gpt-5-nano-2025-08-07
  industries: ['Information Technology', 'Health Care', 'Financials', 'Consumer Discretionary', 'Communication Services', 'Industrials', 'Consumer Staples', 'Energy', 'Utilities', 'Real Estate', 'Materials', 'General Market', 'None']
  dataset_name: danidanou/Bloomberg_Financial_News
  dataset_dir: ../data/
  rss_feeds: ['https://feeds.bloomberg.com/news/news.rss', 'https://feeds.bloomberg.com/markets/news.rss', 'https://feeds.bloomberg.com/business/news.rss', 'https://feeds.bloomberg.com/technology/news.rss', 'https://feeds.bloomberg.com/politics/news.rss', 'https://feeds.bloomberg.com/wealth/news.rss', 'https://feeds.bloomberg.com/economics/news.rss', 'https://feeds.bloomberg.com/green/news.rss', 'https://feeds.bloomberg.com/pursuits/news.rss', 'https://feeds.bloomberg.com/opinion/news.rss', 'https://feeds.bloomberg.com/finance/news.rss', 'https://feeds.bloomberg.com/real-e

# Agent

In [7]:
!pip install feedparser

In [8]:
!pip install langchain langchain-openai

In [9]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import ToolMessage
import json

from tools.mock_data_feed import get_mock_bloomberg_rss_feeds
from tools.data_feed import get_bloomberg_rss_feeds
from tools.data_processor import process_bloomberg_news
from tools.predict_returns import predict_returns_next_day
import nest_asyncio
nest_asyncio.apply()

async def run_agent():
  agent = create_react_agent(
      model="gpt-5-nano",
      tools=[get_bloomberg_rss_feeds, get_mock_bloomberg_rss_feeds, process_bloomberg_news, predict_returns_next_day]
  )
  instructions  = """
  Follow these steps sequentially, passing output from the previous step to the input for the next step.
  1. Fetch MOCK Bloomberg RSS news from the past 1 day.
  2. Then, process the news.
  3. Then, predict returns for the next day.
  4. Finally, return to me a short summary of pos/neg news impacting the market."""#3. Predict returns for the next day.

  last_tool_result = None

  async for chunk in agent.astream({"messages": [{"role": "user", "content": instructions}]}, stream_mode="updates"):
    for step, data in chunk.items():
      print(f"step: {step}")
      print(f"content: {data['messages'][-1].content_blocks}")

      messages = data.get("messages", [])
      if not messages:
          continue

      last_msg = messages[-1]
      if isinstance(last_msg, ToolMessage):
          # Optional: match by tool name
          if last_msg.name == "predict_returns_next_day":
              last_tool_result = last_msg.content
              print("\n=== Extracted tool result ===")
              print(last_tool_result)

  return last_msg.content_blocks[0]["text"], last_tool_result

msg, data = await run_agent()

/tmp/ipython-input-1811803785.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


step: agent
content: [{'type': 'tool_call', 'name': 'get_mock_bloomberg_rss_feeds', 'args': {}, 'id': 'call_2o3C06Nq2svj7Eze0fAUVXQT'}]
step: tools
content: [{'type': 'text', 'text': '[{"Headline": "Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/asian-stocks-to-ebb-as-global-equity-rally-stalls-markets-wrap", "Article": "Asian stocks wobbled at the open Friday as the sharp rebound in global equities over the past week showed signs of losing momentum.", "Date": "2025-11-27T22:29:53+00:00"}, {"Headline": "Gold Poised for Fourth Monthly Gain on Fed Rate-Cut Optimism", "Link": "https://www.bloomberg.com/news/articles/2025-11-27/gold-poised-for-fourth-monthly-gain-on-fed-rate-cut-optimism", "Article": "Gold was steady &mdash; and on track for a fourth monthly gain &mdash; as signs the Federal Reserve will cut rates next month supported the precious metal.", "Date": "2025-11-27T23:44:21+00:00"}, {"Headline": "Oil in

Device set to use cuda:0


Processing 3 news entries...


Industry Classification: 100%|██████████| 1/1 [00:00<00:00,  1.95batch/s]


...

Completed processing 3 entries


Explanation: 100%|██████████| 3/3 [00:13<00:00,  4.56s/it]


step: tools
content: [{'type': 'text', 'text': '[{"Industry":"General Market","Date":"2025-11-27T22:29:53+00:00","News":[{"Headline":"Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap","Date":"2025-11-27T22:29:53+00:00","Link":"https:\\/\\/www.bloomberg.com\\/news\\/articles\\/2025-11-27\\/asian-stocks-to-ebb-as-global-equity-rally-stalls-markets-wrap","Article":"Asian stocks wobbled at the open Friday as the sharp rebound in global equities over the past week showed signs of losing momentum.","SentimentScore":0.0039989622,"Industry":"General Market"}],"ArticleCount":1,"ImpactfulNews":[{"Headline":"Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap","Article":"Asian stocks wobbled at the open Friday as the sharp rebound in global equities over the past week showed signs of losing momentum.","SentimentScore":0.0039989622}],"AvgSentimentScore":0.0039989622,"SentimentScore":0.001944202,"SentimentExplanation":"Near-neutral sentiment with a tiny positive tilt. The art

[*********************100%***********************]  2 of 2 completed
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


OHLCV shape: (62, 10)
OHLCV columns (first few): [('Close', 'SPY'), ('Close', 'XLF'), ('High', 'SPY'), ('High', 'XLF'), ('Low', 'SPY'), ('Low', 'XLF'), ('Open', 'SPY'), ('Open', 'XLF')]
ohlcv_long shape: (124, 4)
model_df shape after OHLCV merge: (3, 20)
Dropped 0 rows due to missing MKT or SentimentScore.
Numeric feature matrix shape: (3, 4)
step: tools
content: [{'type': 'text', 'text': '[{"Industry":"General Market","Date":1764282593000,"News":[{"Headline":"Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap","Article":"Asian stocks wobbled at the open Friday as the sharp rebound in global equities over the past week showed signs of losing momentum.","SentimentScore":0.0039989622,"Date":1764282593000,"Link":"https:\\/\\/www.bloomberg.com\\/news\\/articles\\/2025-11-27\\/asian-stocks-to-ebb-as-global-equity-rally-stalls-markets-wrap","Industry":null}],"ArticleCount":1,"ImpactfulNews":[{"Headline":"Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap","Article":"Asi

In [10]:
print(msg)

Here’s the sequential result and a concise pos/neg news snapshot from the pipeline you asked for.

Step results (summary)
- Mock Bloomberg RSS items (past day):
  - Asian Stocks Ebb as Global Rally Loses Momentum: Markets Wrap
  - Gold Poised for Fourth Monthly Gain on Fed Rate-Cut Optimism
  - Oil in Worst Monthly Run Since 2023 With OPEC+, Ukraine in Focus

- Processed news (key sentiment by piece)
  - General Market: Asian Stocks Ebb
    - Sentiment: near-neutral with a tiny positive tilt (SentimentScore ~ +0.004)
    - Interpretation: Momentum fading; cautious risk appetite; possible choppiness early in the session.
  - Financials: Oil news
    - Sentiment: mildly negative (-0.0089)
    - Interpretation: Energy backdrop weaker; potential headwind if oil persists lower.
  - Financials: Gold news
    - Sentiment: strongly negative (-0.739)
    - Interpretation: Fed rate-cut optimism influencing gold, implying risk-off pressure on financials (lower net interest margins, etc.).

- Pred

In [12]:
import pandas as pd
parsed = pd.DataFrame(json.loads(data))
print(f"\nFinal number of entries: {len(parsed)}")


Final number of entries: 3


In [17]:
parsed.iloc[0]

,0
Industry,General Market
Date,1764282593000
News,[{'Headline': 'Asian Stocks Ebb as Global Rall...
ArticleCount,1
ImpactfulNews,[{'Headline': 'Asian Stocks Ebb as Global Rall...
AvgSentimentScore,0.003999
SentimentScore,0.001944
SentimentExplanation,Near-neutral sentiment with a tiny positive ti...
index_ticker,SPY
feature_date,1764115200000
